# Term Frequency–Inverse Document Frequency

TF-IDF (Term Frequency–Inverse Document Frequency) is a statistical method used in natural language processing and information retrieval to evaluate how important a word is to a document in relation to a larger collection of documents.

TF-IDF combines two components:

### 1. Term Frequency (TF)

Measures how often a word appears in a document. A higher frequency suggests greater importance — if a term appears frequently in a document, it is likely relevant to the document's content.

$$\text{TF}(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}$$

where $f_{t,d}$ is the number of times term $t$ appears in document $d$, and the denominator is the total number of terms in $d$.

### 2. Inverse Document Frequency (IDF)

Reduces the weight of common words across multiple documents while increasing the weight of rare words. If a term appears in fewer documents, it is more likely to be meaningful and specific.

$$\text{IDF}(t, D) = \log \frac{N}{|\{d \in D : t \in d\}|}$$

where $N$ is the total number of documents in the corpus $D$, and the denominator is the number of documents that contain term $t$.

### Combining the two

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

---

This balance allows TF-IDF to highlight terms that are both frequent within a specific document and distinctive across the text document, making it a useful tool for tasks like search ranking, text classification and keyword extraction.


In [2]:
import re

In [68]:
TOKEN = r'\w+(?:\.\w+)*'

def tokenize(text):
    return re.findall(TOKEN, text.lower())

def eval_term_frequency(term, doc):
    tokens = tokenize(doc)
    if not tokens:
        return 0.0
    target = tokenize(term)
    return tokens.count(target[0]) / len(tokens) if target else 0.0

In [69]:
# --- Unit tests for eval_term_frequency ---
# Make the CODE pass these. Don't change the tests to fit the code.
# Run this cell after each edit to your function and watch which ones flip green.

import math

def _approx(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=0, abs_tol=tol)

def run_tf_tests():
    cases = [
        # (term, doc, expected, note)
        ("cat", "the cat sat",               1/3, "basic: 1 of 3 words"),
        ("the", "the cat sat the the",       3/5, "repeated term"),
        ("dog", "the cat sat",               0.0, "term absent -> 0"),
        ("cat", "the cat scattered cats",    1/4, "must NOT match substrings in 'scattered'/'cats'"),
        ("Cat", "cat CAT cat",               1.0, "case-insensitive: 'Cat' matches cat/CAT/cat -> 3/3"),
        ("CAT", "The Cat sat",               1/3, "case-insensitive: term and doc differ in case"),
        ("cat", "cat,cat cat",               1.0, "punctuation: numerator & denominator must agree on tokens -> 3 cats / 3 tokens"),
        ("c++", "i love c++ and c++",        2/5, "term is DATA not regex: 'c++' must not crash or mis-match"),
        ("u.s.a", "u.s.a is big",            1/3, "dots in term must be literal, not regex 'any char'"),
    ]

    passed = 0
    for i, (term, doc, expected, note) in enumerate(cases, 1):
        try:
            got = eval_term_frequency(term, doc)
            ok = _approx(got, expected)
            status = "PASS" if ok else "FAIL"
            passed += ok
            print(f"[{status}] test {i}: tf({term!r}, {doc!r}) = {got}  (expected {expected})  # {note}")
        except Exception as e:
            print(f"[ERROR] test {i}: tf({term!r}, {doc!r}) raised {type(e).__name__}: {e}  # {note}")

    # Edge case: empty document should not crash. Decide your own contract
    # (return 0.0? raise a clear error?) and assert it here once you've decided.
    try:
        got = eval_term_frequency("cat", "")
        print(f"[INFO ] empty-doc: tf('cat', '') = {got}  (you decide: is this the contract you want?)")
    except Exception as e:
        print(f"[INFO ] empty-doc: raised {type(e).__name__}: {e}  (intentional? document your choice)")

    print(f"\n{passed}/{len(cases)} core tests passing")

run_tf_tests()


[PASS] test 1: tf('cat', 'the cat sat') = 0.3333333333333333  (expected 0.3333333333333333)  # basic: 1 of 3 words
[PASS] test 2: tf('the', 'the cat sat the the') = 0.6  (expected 0.6)  # repeated term
[PASS] test 3: tf('dog', 'the cat sat') = 0.0  (expected 0.0)  # term absent -> 0
[PASS] test 4: tf('cat', 'the cat scattered cats') = 0.25  (expected 0.25)  # must NOT match substrings in 'scattered'/'cats'
[PASS] test 5: tf('Cat', 'cat CAT cat') = 1.0  (expected 1.0)  # case-insensitive: 'Cat' matches cat/CAT/cat -> 3/3
[PASS] test 6: tf('CAT', 'The Cat sat') = 0.3333333333333333  (expected 0.3333333333333333)  # case-insensitive: term and doc differ in case
[PASS] test 7: tf('cat', 'cat,cat cat') = 1.0  (expected 1.0)  # punctuation: numerator & denominator must agree on tokens -> 3 cats / 3 tokens
[PASS] test 8: tf('c++', 'i love c++ and c++') = 0.4  (expected 0.4)  # term is DATA not regex: 'c++' must not crash or mis-match
[PASS] test 9: tf('u.s.a', 'u.s.a is big') = 0.333333333333